In [1]:
# --- MÓDULO DE INGESTÃO: VALOR ADICIONADO BRUTO (VAB) DE SERVIÇOS ---

import pandas as pd

# Defino o ponto de entrada para os dados do Setor Terciário.
# Esta variável é vital para capturar o "Efeito Multiplicador" (spillover) 
# da atividade minerária sobre o comércio, logística e serviços locais de Araxá.
caminho_arquivo = r'C:\Users\fabio\TCC\6_VAB_Servicos.csv'

try:
    # Realizo a ingestão do dataset da composição setorial de serviços.
    # skiprows=1: Tratamento padronizado para cabeçalhos do IBGE/SIDRA.
    # encoding='utf-8-sig': Previne erros de decodificação de caracteres latinos (BOM).
    df_vab_servicos = pd.read_csv(
        caminho_arquivo,
        skiprows=1,
        sep=';',
        encoding='utf-8-sig'
    )
    
    # DIAGNÓSTICO ESTRUTURAL INICIAL:
    # Confirmo a leitura correta das dimensões e séries temporais do IBGE.
    print("--- Visualização do Dataset do Setor Terciário (VAB Serviços) ---")
    print(df_vab_servicos.head())
    
    print("\n--- Inventário de Variáveis Disponíveis ---")
    print(df_vab_servicos.columns.tolist())

except Exception as e:
    print(f"ERRO DE INGESTÃO: Falha crítica ao carregar a matriz de VAB de Serviços: {e}")

--- TABELA CARREGADA ---
  Sigla   Código     Município              1996              1999  \
0    AC  1200013    Acrelândia  14624,1544717856   10799,543537354   
1    AC  1200054  Assis Brasil  12556,3154886708  6072,06123391626   
2    AC  1200104     Brasiléia  92197,5227897897  39295,5576437541   
3    AC  1200138        Bujari   25540,522062857   9114,0518638641   
4    AC  1200179      Capixaba  17981,6002806603  6889,77501610443   

               2000              2001              2002              2003  \
0  14883,3745805536  19008,6701084132   9939,9030994852   10357,539757247   
1  7442,81601578725  6872,99111239125   4216,4268820684   4172,7328627937   
2  44918,7604131437  38561,7562047882  28323,6480927391  30210,2466429594   
3  10027,5974346761  10458,6274468982   4603,4373945533   5061,6091141096   
4  8973,36780793277   11806,109164229   6331,1004086513   6392,6697590297   

               2004  ...              2013              2014  \
0  10818,8814087407  ...  2

In [2]:
# --- ETAPA 2: NORMALIZAÇÃO E TRANSPOSIÇÃO SETORIAL (VAB SERVIÇOS) ---

# Nesta etapa, replico o pipeline de higienização e reestruturação de dados 
# para o Setor Terciário. O objetivo é converter a base horizontal ('Wide') 
# em um painel vertical ('Long'), preparando o indicador de serviços locais 
# para a junção com a base mestre do Controle Sintético.

print("Iniciando a higienização dos atributos do Setor de Serviços...")

# 1. TRATAMENTO DE ATRIBUTOS RESIDUAIS
# Isolo e removo colunas fantasmas geradas por delimitadores vazios 
# ao final das linhas no padrão de exportação do SIDRA/IBGE.
try:
    df_vab_servicos_limpo = df_vab.drop(columns=['Unnamed: 27'])
    print("Sucesso: Coluna residual 'Unnamed: 27' removida do dataset de Serviços.")
except KeyError:
    print("Aviso: Estrutura do Setor Terciário já normalizada; prosseguindo.")
    df_vab_servicos_limpo = df_vab.copy()

# 2. REESTRUTURAÇÃO PARA PAINEL DE DADOS (MELT)
# Aplico o reshaping na matriz para que a variável métrica 'VAB Serv.' 
# seja distribuída de forma cronológica por município.

# 2a. Definição das Chaves Primárias (Fixas)
dimensoes_id = ['Sigla', 'Código', 'Município']

# 2b. Execução da Transposição (Wide to Long)
df_vab_servicos_final = df_vab_servicos_limpo.melt(
    id_vars=dimensoes_id,
    var_name='Ano',
    value_name='VAB Serv.'
)

# 3. VALIDAÇÃO TÉCNICA DO PAINEL DE SERVIÇOS
# Inspeciono o cabeçalho e o rodapé para auditar a continuidade temporal.
print("\n--- Diagnóstico: Painel de Valor Adicionado (VAB Serviços) ---")
print("Cabeçalho (Início da Série Histórica):")
print(df_vab_servicos_final

Coluna 'Unnamed: 27' removida com sucesso.

--- Tabela 'Derretida' (Formato Longo) ---
  Sigla   Código     Município   Ano         VAB Serv.
0    AC  1200013    Acrelândia  1996  14624,1544717856
1    AC  1200054  Assis Brasil  1996  12556,3154886708
2    AC  1200104     Brasiléia  1996  92197,5227897897
3    AC  1200138        Bujari  1996   25540,522062857
4    AC  1200179      Capixaba  1996  17981,6002806603
       Sigla   Código       Município   Ano         VAB Serv.
134323    TO  1721208  Tocantinópolis  2021  51903,3688767557
134324    TO  1721257        Tupirama  2021   13490,730659228
134325    TO  1721307      Tupiratins  2021  2121,63912962599
134326    TO  1722081    Wanderlândia  2021  13373,5620296271
134327    TO  1722107         Xambioá  2021  32070,1275785336


In [3]:
# --- ETAPA 3: SERIALIZAÇÃO E PERSISTÊNCIA DO VAB DE SERVIÇOS (OUTPUT) ---

# Concluo o módulo do Setor Terciário com a exportação do painel consolidado.
# Este arquivo é a peça-chave para testar a hipótese de transbordamento econômico
# (spillover effect) da mineração para o comércio e serviços da região.

# 1. DEFINIÇÃO DO DIRETÓRIO DE SAÍDA
# Padronizo a nomenclatura do arquivo de saída para garantir a leitura 
# sequencial no futuro script de integração mestre (Master Merge).
caminho_saida_csv = r'C:\Users\fabio\TCC\6_VAB_SERV_FINAL.csv'

# 2. EXPORTAÇÃO E INTEGRIDADE DO DATASET
# Utilizo index=False para suprimir os índices nativos do Pandas, gerando um 
# CSV estruturado e otimizado para a ingestão em modelos econométricos (Stata/R).
df_vab_servicos_final.to_csv(
    caminho_saida_csv,
    index=False
)

print(f"--- PROTOCOLO DE EXPORTAÇÃO CONCLUÍDO ---")
print(f"Dataset do Setor Terciário (Serviços) exportado com sucesso em:")
print(caminho_saida_csv)

--- SUCESSO! ---
Seu arquivo de população consolidado foi salvo em:
C:\Users\fabio\TCC\6_VAB_SERV_FINAL.csv
